# Presto, frozen encoder (128-d embeddings)

Presto (Lightweight, Pre-trained Transformers for Remote Sensing Timeseries, Tseng et al. 2023) is a **pixel-based transformer** designed for multi-sensor satellite time series. It takes 12 monthly timesteps of 17 bands (S1, S2, ERA5, SRTM, NDVI) plus the pixel’s latitude/longitude and produces a single 128-dimensional embedding per pixel.

Key properties:

- Tiny: only **~0.4M parameters** (2 transformer layers, 8 heads, embedding size 128). Compare to ViT-based models with 100M+ params.
- **Multi-modal**: ingests S1 SAR, S2 optical, ERA5 climate, SRTM elevation, and NDVI simultaneously.
- **Location-aware**: the pixel’s [lat, lon] is encoded into each token, so the model can learn geographic priors.
- **Missing-data robust**: pre-trained with structured masking (random bands, random timesteps, entire sensor groups) so it handles degraded/absent inputs naturally.
- Originally pre-trained on **21.5M global pixel samples** from a 2-year period, using a masked-autoencoding objective.

In this repo it serves as the lightweight frozen baseline: the cheapest encoder to run, and the model WorldCereal actually deploys.


## What Presto expects as input

Presto takes **five separate tensors** sent to the same `forward()` call. They enter the model through different pathways and get fused inside the transformer:

```python
encoder.forward(x, dynamic_world, latlons, mask, month, eval_task=True)
```

B = number of samples in the batch (e.g. 2048). 12 = number of timesteps (fixed in our pipeline).

| Tensor | Shape | Pathway inside model |
|--------|-------|---------------------|
| **x** | `(B, 12, 17)` | The band data. Each sensor group (S1, S2, ERA5, etc.) is projected to the embedding space by a **separate learned linear layer** before the transformer. The 17-band column order is fixed (see table below). |
| **dynamic_world** | `(B, 12)` | Land-cover class (0–8, or 9 = missing). Goes through an **embedding lookup table** (class index → vector). We always pass 9 since we don’t have DW data. |
| **latlons** | `(B, 2)` | [lat, lon] of the pixel. Added as a **positional bias** so the model can learn geographic priors. |
| **mask** | `(B, 12, 17)` | 1 = that band at that timestep is missing. **Modulates self-attention**: masked positions are ignored. This is how Presto knows what data is absent versus just zero-valued. |
| **month** | `(B,)` | Starting month (0–11) of the time series. Determines how **positional encodings cycle** over the 12 timesteps. |
| output | `(B, 128)` | One 128-d embedding per sample (mean of all output tokens, normalised). |

### The 17 bands in order (index 0–16)

This is the exact column layout of the `(B, 12, 17)` array that Presto expects:

| Idx | Band | Source | Sensor/Product | What it measures |
|:---:|------|--------|----------------|------------------|
| 0 | VV | Sentinel-1 | C-band SAR (radar) | Backscatter, vertical transmit/vertical receive |
| 1 | VH | Sentinel-1 | C-band SAR (radar) | Backscatter, vertical transmit/horizontal receive |
| 2 | B2 | Sentinel-2 | MSI (10 m) | Blue |
| 3 | B3 | Sentinel-2 | MSI (10 m) | Green |
| 4 | B4 | Sentinel-2 | MSI (10 m) | Red |
| 5 | B5 | Sentinel-2 | MSI (20 m) | Red Edge 1 |
| 6 | B6 | Sentinel-2 | MSI (20 m) | Red Edge 2 |
| 7 | B7 | Sentinel-2 | MSI (20 m) | Red Edge 3 |
| 8 | B8 | Sentinel-2 | MSI (10 m) | Near-infrared (broad) |
| 9 | B8A | Sentinel-2 | MSI (20 m) | Near-infrared (narrow) |
| 10 | B11 | Sentinel-2 | MSI (20 m) | Shortwave infrared 1 |
| 11 | B12 | Sentinel-2 | MSI (20 m) | Shortwave infrared 2 |
| 12 | temperature_2m | ERA5 | Climate reanalysis | Air temperature at 2 m (Kelvin) |
| 13 | total_precipitation | ERA5 | Climate reanalysis | Total precipitation (metres) |
| 14 | elevation | SRTM | Digital elevation model | Terrain elevation (metres) |
| 15 | slope | SRTM | DEM derivative | Terrain slope (degrees) |
| 16 | NDVI | Computed | Vegetation index | `(B8-B4)/(B8+B4)` — plant greenness |

### Per-band normalisation

Each band is normalised with fixed (learned) constants before entering the model:

| Band group | Formula | Effect |
|---|---|---|
| S1 (VV, VH) | `(x + 25) / 25` | Linear shift to ~[0, 2] range |
| S2 (B2–B12) | `x / 10000` | Raw DN (0–10000) to [0, 1] reflectance |
| temperature_2m | `(x - 272.15) / 35` | Kelvin to centred Celsius-like scale |
| total_precipitation | `x / 0.03` | Raw m to ~[0, 1+] |
| elevation | `x / 2000` | Metres to ~[0, 1] (max ~2000 m) |
| slope | `x / 50` | Degrees to ~[0, 1] (max ~50°) |
| NDVI | `(B8 - B4) / (B8 + B4)` | Re-computed from **already-normalised** B8/B4 |

Note: slope exists in Presto’s band list but is **never available** in our pipeline — our code does not load it from either CropHarvest or EuroCropsML, so it is always marked as missing in the mask.

### Caveats
- We don't have `slope` (SRTM) -> always masked. `B9` is excluded from Presto's 17 bands by design.
- `month` comes from `doy[:, 0]`; our CropHarvest dates are synthetic (monthly), EuroCropsML real.
- Normalization constants are copied verbatim from Presto's source and cross-checked to match
  `construct_single_presto_input` exactly (max abs diff 0).

In [15]:
# Bootstrap: find repo root and load a model module by path (src has no __init__ files).
# Use a kernel that has the deps (numpy, torch, and the model's package).
import sys, importlib.util
from pathlib import Path
from types import SimpleNamespace
import numpy as np

REPO = Path.cwd()
while not (REPO / 'src').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

def load(name, rel):
    spec = importlib.util.spec_from_file_location(name, REPO / rel)
    m = importlib.util.module_from_spec(spec); sys.modules[name] = m; spec.loader.exec_module(m)
    return m

In [16]:
presto = load('presto_mod', 'src/models/presto.py')
print('PRESTO_BANDS (17):')
for i, b in enumerate(presto.PRESTO_BANDS):
    print(f'  {i:2d}  {b:<13} add={presto.PRESTO_ADD[i]:>9}  divide={presto.PRESTO_DIVIDE[i]:>8}')
print('embedding dim:', presto.PRESTO_EMBEDDING_DIM, '| dynamic_world missing class:', presto.DYNAMIC_WORLD_MISSING)

PRESTO_BANDS (17):
   0  VV            add=     25.0  divide=    25.0
   1  VH            add=     25.0  divide=    25.0
   2  B2            add=      0.0  divide= 10000.0
   3  B3            add=      0.0  divide= 10000.0
   4  B4            add=      0.0  divide= 10000.0
   5  B5            add=      0.0  divide= 10000.0
   6  B6            add=      0.0  divide= 10000.0
   7  B7            add=      0.0  divide= 10000.0
   8  B8            add=      0.0  divide= 10000.0
   9  B8A           add=      0.0  divide= 10000.0
  10  B11           add=      0.0  divide= 10000.0
  11  B12           add=      0.0  divide= 10000.0
  12  temperature   add=-272.1499938964844  divide=    35.0
  13  precipitation add=      0.0  divide=0.029999999329447746
  14  elevation     add=      0.0  divide=  2000.0
  15  slope         add=      0.0  divide=    50.0
  16  NDVI          add=      0.0  divide=     1.0
embedding dim: 128 | dynamic_world missing class: 9


In [17]:
# A tiny synthetic Benchmark (duck-typed): just the attributes an encoder reads.
# This matches what src/dataio/get_input.py produces, but needs no data on disk.
N, T = 4, 12
rng = np.random.default_rng(0)
bench = SimpleNamespace(
    s2=(rng.random((N, T, 11)) * 5000).astype('float32'),   # B2..B12, NDVI (reflectance-ish)
    s1=(rng.random((N, T, 2)) * -15).astype('float32'),     # VV, VH (dB-ish)
    climate=(rng.random((N, T, 3)) * [300, 0.01, 500]).astype('float32'),  # temp(K), precip(m), elev(m)
    s2_mask=np.ones((N, T), 'float32'),
    s1_mask=np.ones((N, T), 'float32'),
    climate_mask=np.ones((N, T), 'float32'),
    doy=np.tile(np.linspace(15, 350, T), (N, 1)).astype('float32'),
    latlon=np.array([[11.0, 79.0]] * N, 'float32'),  # Cauvery Delta-ish (lat, lon)
    s2_bands=['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12','NDVI'],
    s1_bands=['VV','VH'],
    climate_bands=['temperature','precipitation','elevation'],
)
print('synthetic bench:', N, 'samples x', T, 'timesteps')

synthetic bench: 4 samples x 12 timesteps


## Wrapper mapping: `Benchmark -> Presto inputs`
`to_presto_inputs` places each Benchmark band into its `PRESTO_BANDS` slot, normalizes, recomputes
NDVI, and builds the per-band mask. Watch `slope` (index 15) stay fully masked.

In [18]:
enc = presto.PrestoEncoder(device='cpu')
x, mask, dw, latlons, months = enc.to_presto_inputs(bench)
print('x           ', x.shape)         # (N, T, 17)
print('mask        ', mask.shape, '| slope always missing:', bool((mask[:, :, 15] == 1).all()))
print('dynamic_world', dw.shape, '| all == 9:', bool((dw == 9).all()))
print('latlons     ', latlons.shape, '| months', months.tolist())
print('normalized B8 range: %.3f .. %.3f' % (x[:, :, 8].min(), x[:, :, 8].max()))

x            (4, 12, 17)
mask         (4, 12, 17) | slope always missing: True
dynamic_world (4, 12) | all == 9: True
latlons      (4, 2) | months [0, 0, 0, 0]
normalized B8 range: 0.007 .. 0.497


## Forward pass → embedding
Loads the real pretrained weights (cached) and returns `(N, 128)`.

In [19]:
emb = enc.encode(bench)
print('embedding:', emb.shape, '| finite:', bool(np.isfinite(emb).all()))

embedding: (4, 128) | finite: True
